# Aula 05 · Newton e secante

Esta aula apresenta o [capítulo 5 do site](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/). A ideia central: **seguir a inclinação da curva leva à raiz em pouquíssimas iterações** — com a tangente, os algarismos certos dobram a cada passo. O preço é que o método pode se perder, e é preciso saber reconhecer isso.

**Ao fim da aula você consegue:**

1. deduzir a fórmula de Newton-Raphson pela reta tangente e aplicá-la à mão;
2. usar Newton com a derivada numérica do capítulo 2, e a secante, que não precisa de derivada;
3. comparar a velocidade dos quatro métodos de raízes num gráfico de erro;
4. reconhecer quando Newton anda em círculos ou é atirado para longe.

**Roteiro:** 🧩 · 1. 🧑‍🏫 Newton · 2. sem derivar à mão · 3. 🧑‍🏫 secante · 4. comparando · 5. quando Newton falha · 6. outra área · 🎯 prática · 🧩 a boia · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Oceanografia — boia de monitoramento.**
>
> *Uma boia esférica de 1 m de diâmetro vai ancorada na costa com sensores de onda
> e um rádio que transmite os dados para terra. O pacote de equipamentos (bateria,
> painel solar, sensores, rádio) pesa **150 kg**. O rádio só tem alcance se a antena
> ficar alta: o **topo da boia precisa ficar pelo menos 40 cm acima da água**. A
> engenheira pergunta: "**com essa carga, quanto da boia fica fora d'água?**"*

Pelo princípio de Arquimedes, a boia afunda até o empuxo igualar o peso. A
equação do volume de uma calota esférica não se deixa isolar para a profundidade:
é uma raiz. No fim da aula, você a acha por Newton.

## 1. Newton-Raphson

📖 [capítulo 5 · Newton-Raphson](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#newton-raphson)

### 🧑‍🏫 No quadro — o método de Newton-Raphson

Caderno de papel aberto. No quadro:

1. no chute $x_i$, a reta tangente, com inclinação $f'(x_i)$;
2. onde a tangente cruza o zero: $f'(x_i) = \dfrac{f(x_i)}{x_i - x_{i+1}}$;
3. isolar $x_{i+1}$;
4. por que o erro novo é proporcional ao **quadrado** do anterior;
5. o caso $f(x) = x^2 - a$: a raiz quadrada da calculadora.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ x_{i+1} = x_i - \frac{f(x_i)}{f'(x_i)} $$

Convergência **quadrática**: $E_{i+1} \approx C\,E_i^2$ — os algarismos certos
dobram a cada iteração. Para $x^2 - a$: $x_{i+1} = \frac{1}{2}\left(x_i + \frac{a}{x_i}\right)$.

</details>

**✍️ Passo 1.** Com `x = 1.0`, faça **um** passo de Newton para $x^2 - 2$ (derivada $2x$) e imprima `x`.

In [ ]:
# ✍️ passo 1

**Preveja:** o novo `x` fica acima ou abaixo de $\sqrt{2} = 1{,}4142$?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`1.5`: acima. A tangente em $x = 1$ cruza o zero em $1 + 1/2$.

</details>

**✍️ Passo 2.** Agora 5 passos num laço, imprimindo `x` e o erro `abs(x - np.sqrt(2))` com `:.1e`.

In [ ]:
# ✍️ passo 2

**Preveja:** o que acontece com o **expoente** do erro a cada iteração?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

$10^{-2}$, $10^{-3}$, $10^{-6}$, $10^{-12}$, zero: o expoente **dobra**.
Em 5 iterações, os 16 algarismos do `float` estão certos.

📖 [capítulo 5 · Newton-Raphson](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#newton-raphson)

</details>

### 🎯 Sua vez — Um passo de Newton

Escreva `passo_newton(f, df, x)`, que devolve o próximo ponto de Newton a partir de `x`.

In [ ]:
def passo_newton(f, df, x):
    # sua solução aqui
    pass

In [ ]:
def cubo_menos_20(x):
    return x**3 - 20


def derivada_cubo(x):
    return 3 * x**2


confere(passo_newton, [
    ((cubo_menos_20, derivada_cubo, 3.0), 2.740740740740741),
])

<details>
<summary><b>💡 Dica</b></summary>

Uma linha: a fórmula do quadro.

</details>

## 2. Newton sem derivar à mão

Derivar a fórmula do paraquedista em relação a $m$ à mão é trabalhoso. A diferença
central do capítulo 2 faz isso sem derivar nada.

📖 [capítulo 5 · Newton sem derivar à mão](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#newton-sem-derivar-a-mao)

In [ ]:
# 📦 dados prontos — só rode esta célula
g = 9.81
c = 0.25
t = 4


def f(m):
    v = np.sqrt(g * m / c) * np.tanh(np.sqrt(g * c / m) * t)
    return v - 36

**✍️ Passo 3.** Escreva `derivada(f, x)` com a diferença central (`h = 1e-5`) e faça Newton na `f` do paraquedista a partir de `x = 100.0`, 6 iterações, imprimindo `x` e `f(x)`.

In [ ]:
# ✍️ passo 3

**Preveja:** em quantas iterações `x` para de mudar na sexta casa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Em **4**: 142,737633 kg. A bisseção precisou de 11 iterações para 4
algarismos. Nas últimas, $f(x)$ fica em $10^{-15}$: é o arredondamento, e não
há mais o que melhorar.

📖 [capítulo 5 · Newton sem derivar à mão](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#newton-sem-derivar-a-mao)

</details>

## 3. A secante

📖 [capítulo 5 · A secante](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#a-secante)

### 🧑‍🏫 No quadro — o método da secante

Caderno de papel aberto. No quadro:

1. sem derivada: a inclinação da reta entre os **dois últimos** pontos;
2. trocar $f'(x_i)$ por essa inclinação na fórmula de Newton;
3. a diferença para a falsa posição: aqui não se guarda troca de sinal.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ x_{i+1} = x_i - \frac{f(x_i)\,(x_{i-1} - x_i)}{f(x_{i-1}) - f(x_i)} $$

Precisa de **dois chutes**, não precisa de derivada nem de troca de sinal. Ordem
de convergência ≈ 1,6.

</details>

**✍️ Passo 4.** Com `x_ant = 50.0` e `x = 60.0`, faça 8 iterações da secante na `f` do paraquedista, imprimindo `x`. Cuidado com a ordem da atualização: calcule o `novo`, depois `x_ant = x` e só então `x = novo`.

In [ ]:
# ✍️ passo 4

**Preveja:** os dois chutes estão do mesmo lado da raiz ($f(50)$ e $f(60)$ são negativos). O método funciona?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Funciona: 142,737633 kg em 7 iterações. A secante não precisa de troca de
sinal — e por isso também não tem a garantia da bisseção.

📖 [capítulo 5 · A secante](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#a-secante)

</details>

> ⚠️ **Armadilha.** Fazer `x = novo` **antes** de `x_ant = x` perde o ponto anterior: os dois
ficam iguais e a próxima conta divide por zero.

### 🎯 Sua vez — O ponto da secante

Escreva `ponto_secante(f, x_ant, x)`, que devolve o próximo ponto da secante.

In [ ]:
def ponto_secante(f, x_ant, x):
    # sua solução aqui
    pass

In [ ]:
confere(ponto_secante, [
    ((cubo_menos_20, 2.0, 3.0), 2.6315789473684212),
])

<details>
<summary><b>💡 Dica</b></summary>

É a fórmula da falsa posição com $a$ no lugar de $x_{i-1}$ e $b$ no lugar de $x_i$.

</details>

## 4. Comparando os quatro métodos

📖 [capítulo 5 · Comparando os quatro métodos](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#comparando-os-quatro-metodos)

**✍️ Passo 5.** Faça Newton em $x^3 - 20$ (derivada $3x^2$) a partir de `x = 3.0`, imprimindo o erro `abs(x - 20 ** (1 / 3))` a cada iteração. Quantas iterações até o erro ficar abaixo de $10^{-10}$?

In [ ]:
# ✍️ passo 5

**Preveja:** a bisseção, partindo de $[2, 3]$, precisa de 32 iterações para isso. E Newton?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**4 iterações.** A secante precisa de 5, a falsa posição de 10 e a bisseção de
32. O gráfico do capítulo mostra os quatro: as curvas da secante e de Newton
**entortam para baixo** — cada iteração ganha mais que a anterior.

📖 [capítulo 5 · Comparando os quatro métodos](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#comparando-os-quatro-metodos)

</details>

## 5. Quando Newton falha

📖 [capítulo 5 · Quando Newton falha](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#quando-newton-falha)

**✍️ Passo 6.** Aplique Newton a $f(x) = x^3 - 2x + 2$ (derivada $3x^2 - 2$) a partir de `x = 0.0`, 6 iterações, imprimindo `x`.

In [ ]:
# ✍️ passo 6

**Preveja:** vai convergir para a raiz (perto de −1,77)?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: `1.0`, `0.0`, `1.0`, `0.0`... Newton **anda em círculos**, para sempre e
sem mensagem de erro. A tangente em 0 aponta para 1, e a tangente em 1 aponta
de volta para 0.

📖 [capítulo 5 · Quando Newton falha](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#quando-newton-falha)

</details>

> ⚠️ **Armadilha.** Um Newton sem número máximo de iterações e sem conferir $f(x)$ no fim pode
rodar para sempre — ou devolver um número que não é raiz. Sempre as duas
proteções: `max_iter` e olhar se $f(x) \approx 0$.

## 6. Mesmo método, outra área

**Astronomia e telecom.** Onde está um satélite numa órbita muito achatada (tipo
Molniya, $e = 0{,}74$, período de 12 h)? A equação de Kepler, $M = E - e\sin E$,
liga o tempo ($M = 2\pi t/T$) à posição ($E$). Não há fórmula para $E$.

📖 [capítulo 5 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#mesmo-metodo-outra-area)

**✍️ Passo 7.** Com `e = 0.74` e `M = 2 * np.pi * 1 / 12` (uma hora depois do perigeu), faça Newton em $f(E) = E - e\sin E - M$, com $f'(E) = 1 - e\cos E$, a partir de `E = M`. Imprima `E` a cada iteração.

In [ ]:
# ✍️ passo 7

**Preveja:** quantas iterações até `E` parar de mudar na sexta casa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Umas 5 ou 6: $E = 1{,}218030$ rad. Com ele, a distância ao centro da Terra
é $a(1 - e\cos E)$, e dá para apontar a antena. Os receptores de GPS resolvem
essa equação para cada satélite, várias vezes por segundo.

📖 [capítulo 5 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

Retoma o bloco *1. Newton-Raphson*.
📖 [capítulo 5 · Newton-Raphson](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/#newton-raphson)

### 🎯 Sua vez — A raiz quadrada da calculadora

Escreva `raiz_quadrada(a)`, que calcula $\sqrt{a}$ pelo método babilônico,
$x_{i+1} = (x_i + a/x_i)/2$, a partir de $x_0 = a$, e para quando a mudança
for menor que $10^{-12}$ vezes o valor novo (no máximo 100 iterações).

In [ ]:
def raiz_quadrada(a):
    # sua solução aqui
    pass

In [ ]:
confere(raiz_quadrada, [
    ((2,), 1.4142135623730951),
    ((144,), 12.0),
    ((1e10,), 100000.0),
])

<details>
<summary><b>💡 Dica</b></summary>

O laço do 🎯 da raiz cúbica da lista, com outra fórmula. Guarde `novo` antes de comparar.

</details>

## 🧩 Resolvendo o problema

> *"**Com essa carga, quanto da boia fica fora d'água?**"* — a engenheira.

A boia afunda até uma profundidade $h$ em que a água deslocada pesa o mesmo que a
boia com a carga. A célula 📦 tem a função `sobra_de_empuxo(h, carga)`, que vale
zero exatamente nessa profundidade.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Boia esférica de sinalização: raio 0,5 m, casca + lastro com 209,4 kg.
R = 0.5                     # raio (m)
RHO_AGUA = 1025             # densidade da água do mar (kg/m³)
MASSA_BOIA = 209.4          # kg
BORDO_MINIMO = 0.40         # a antena precisa de 40 cm do topo da boia até a água


def volume_submerso(h):
    """Volume (m³) de uma esfera de raio R mergulhada até a altura h."""
    return np.pi * h**2 * (3 * R - h) / 3


def sobra_de_empuxo(h, carga):
    """Empuxo (em kg de água deslocada) menos o peso (boia + carga)."""
    return RHO_AGUA * volume_submerso(h) - (MASSA_BOIA + carga)

### 🎯 Sua vez — A boia afunda quanto?

Escreva `afundamento(carga)`, que acha a raiz de `sobra_de_empuxo(h, carga)`
por **Newton com derivada numérica** (diferença central, passo `1e-6`), a
partir de `h = R`, e para quando a mudança for menor que $10^{-10}$ (no
máximo 50 iterações).

In [ ]:
def afundamento(carga):
    # sua solução aqui
    pass

In [ ]:
confere(afundamento, [
    ((0,), 0.42624524465823915),
    ((150,), 0.6151433537255505),
])

<details>
<summary><b>💡 Dica</b></summary>

É o passo 3, com `sobra_de_empuxo(h, carga)` no lugar de `f(m)`. O chute
`h = R` (a boia afundada até a metade) é bom: está perto e $f'$ não é zero lá.

</details>

A resposta para a engenheira:

In [ ]:
h = afundamento(150)
if h is not None:
    print("profundidade:", h, "m")
    print("bordo livre (topo acima da água):", 2 * R - h, "m  (mínimo:", BORDO_MINIMO, "m)")

<details>
<summary><b>▶ O que os números dizem</b></summary>

Com 150 kg de carga, a boia afunda **0.615 m**, e o topo fica **0.385 m**
acima da água: **abaixo** dos 40 cm exigidos pelo rádio. Vazia, ela afundaria só
0.426 m.

A engenheira tem três saídas: aliviar a carga, subir a antena num mastro ou usar
uma boia maior. Para decidir quanta carga ainda cabe, o problema se inverte: dado o
bordo livre de 40 cm ($h = 0{,}6$ m), a carga sai por uma conta direta, sem raiz
nenhuma — nem todo problema precisa de método numérico.

</details>

## 📋 A lista

Abra a [Lista 05](https://lacouth.github.io/metodos_telecom-site/listas/lista05/). O **Exercício 01** é à mão (✏️): Newton e secante para
$\sqrt[3]{10}$. Comece por ele, no papel.

**a)** No primeiro passo de Newton a partir de $x_0 = 2$, quanto valem $f(2)$ e $f'(2)$?

<details>
<summary><b>▶ Resposta</b></summary>

$f(2) = 8 - 10 = -2$ e $f'(2) = 3 \cdot 4 = 12$. Então $x_1 = 2 - (-2)/12 = 2{,}166667$.

</details>

Termine a tabela e siga para o **Exercício 02**, Newton como função.

## 🚪 Antes de sair

**1.** Newton converge mais rápido que a bisseção. Por que, então, alguém ainda usaria a bisseção?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque a bisseção **sempre** converge (se há troca de sinal) e diz de antemão quantas iterações vai gastar. Newton pode divergir ou andar em círculos. Os métodos profissionais começam seguros e terminam rápidos.

</details>

**2.** Que informação a secante usa no lugar da derivada?

<details>
<summary><b>▶ Resposta da 2</b></summary>

A inclinação da reta entre os dois últimos pontos: uma diferença finita regressiva, como no capítulo 2.

</details>

**3.** Por que o chute $h = R$ é bom para a boia, e $h = 0$ seria ruim?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Em $h = 0$ a derivada do volume submerso é zero (a esfera toca a água num ponto só): Newton dividiria por zero. Em $h = R$ a derivada é a maior possível e o chute está perto da resposta.

</details>

## 🏠 Para casa

- Refaça no papel três passos de Newton para $\sqrt[3]{10}$ **sem olhar**.
- Termine a [Lista 05](https://lacouth.github.io/metodos_telecom-site/listas/lista05/).
- Leia o começo do [capítulo 6](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/06-otimizacao/): o ponto mais
  alto de uma curva também é uma raiz — da derivada.